# Task 1: Classical baseline
Language identification → Dakshina normalization → TF-IDF + Random Forest sentiment classifier, on Partition B.

- **Inputs:** `data/partition_b_*.json` and `data/dakshina_norm_dict.json` (from `data_prep`)
- **Outputs:** `results/task1_metrics.json`, plus `data/partition_b_*_processed.json` (predicted language tags and normalized text, used by Tasks 2 and 3)
- **Runtime:** CPU, under a minute.

## Setup

In [ ]:
# Work in a folder on Google Drive so all notebooks share data/ and results/
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/NLP_Group17"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

## Configuration

In [ ]:
import os
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import f1_score, classification_report, accuracy_score

SEED = 42
DATA_DIR = "data"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

## Loading the splits

In [ ]:
def load_split(name):
    with open(os.path.join(DATA_DIR, f"partition_b_{name}.json"), "r", encoding="utf-8") as f:
        return json.load(f)

## Normalization
Tokens the language-ID tagger does not label as English are looked up in the Dakshina dictionary. Tokens without an entry keep their original form, so normalization changes nothing else.

In [ ]:
def normalize_tokens(tokens, lid_tags, dakshina_dict):
    """
    Map Romanized-Hindi spelling variants to their Dakshina canonical form.
    Only tokens the LID tagger did not tag as English are looked up (case-insensitive).
    Tokens with no dictionary entry keep their original casing, so the only change
    from raw text is the Dakshina mapping itself (MuRIL is a cased model, so
    lowercasing everything would confound the Task 2 raw-vs-normalized comparison).
    """
    norm_toks = []
    for tok, tag in zip(tokens, lid_tags):
        low = tok.lower()
        if tag != "Eng" and low in dakshina_dict:
            norm_toks.append(dakshina_dict[low])
        else:
            norm_toks.append(tok)
    return norm_toks

## Pipeline
1. **Language ID:** character 1–4-gram TF-IDF + logistic regression (L2, C = 1.0), trained per token on the gold `Hin` / `Eng` / `O` tags.
2. **Normalization** of every split using the *predicted* tags.
3. **Sentiment classifier:** word + character 1–3-gram TF-IDF + Random Forest (300 trees, max depth 20, at least 5 samples per leaf), scored by macro-F1 on the test split.

In [ ]:
def main():
    train_data = load_split("train")
    val_data = load_split("validation")
    test_data = load_split("test")

    with open(os.path.join(DATA_DIR, "dakshina_norm_dict.json"), "r", encoding="utf-8") as f:
        dakshina_dict = json.load(f)

    # 1. Train Token-Level Language ID (LID) Classifier: char n-gram (1-4) TF-IDF + LogisticRegression(L2, C=1.0)
    print("--- Step 1: Training Token-Level LID Classifier ---")
    train_tokens = [tok for item in train_data for tok in item["tokens"]]
    train_tags = [tag for item in train_data for tag in item["lang_tags"]]

    val_tokens = [tok for item in val_data for tok in item["tokens"]]
    val_tags = [tag for item in val_data for tag in item["lang_tags"]]

    test_tokens = [tok for item in test_data for tok in item["tokens"]]
    test_tags = [tag for item in test_data for tag in item["lang_tags"]]

    lid_vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 4), lowercase=False)
    X_lid_train = lid_vec.fit_transform(train_tokens)
    X_lid_val = lid_vec.transform(val_tokens)
    X_lid_test = lid_vec.transform(test_tokens)

    lid_clf = LogisticRegression(penalty="l2", C=1.0, max_iter=1000, random_state=SEED, n_jobs=-1)
    lid_clf.fit(X_lid_train, train_tags)

    lid_test_preds = lid_clf.predict(X_lid_test)
    lid_acc = accuracy_score(test_tags, lid_test_preds)
    lid_f1 = f1_score(test_tags, lid_test_preds, average="macro")
    print(f"Token-level LID Test Accuracy: {lid_acc:.4f} | LID Macro-F1: {lid_f1:.4f}")

    # Annotate splits with predicted LID tags & Dakshina-normalized text (saved for Task 2 & Task 3)
    for split_name, split_data in [("train", train_data), ("validation", val_data), ("test", test_data)]:
        flat_toks = [tok for item in split_data for tok in item["tokens"]]
        flat_preds = lid_clf.predict(lid_vec.transform(flat_toks))
        idx = 0
        for item in split_data:
            k = len(item["tokens"])
            pred_tags = list(flat_preds[idx : idx + k])
            idx += k
            item["pred_lang_tags"] = pred_tags
            norm_toks = normalize_tokens(item["tokens"], pred_tags, dakshina_dict)
            item["norm_tokens"] = norm_toks
            item["norm_text"] = " ".join(norm_toks)

        with open(os.path.join(DATA_DIR, f"partition_b_{split_name}_processed.json"), "w", encoding="utf-8") as f:
            json.dump(split_data, f, ensure_ascii=False, indent=2)

    # 2. Downstream Sentiment Classifier: Word + Char (1-3) TF-IDF + Random Forest (300 trees, max_depth=20, min_samples_leaf=5)
    print("\n--- Step 2: Training Downstream Random Forest Sentiment Classifier ---")
    X_train_text = [item["norm_text"] for item in train_data]
    y_train = [LABEL2ID[item["sentiment"]] for item in train_data]

    X_val_text = [item["norm_text"] for item in val_data]
    y_val = [LABEL2ID[item["sentiment"]] for item in val_data]

    X_test_text = [item["norm_text"] for item in test_data]
    y_test = [LABEL2ID[item["sentiment"]] for item in test_data]

    tfidf_union = FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 3), max_features=25000, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(1, 3), max_features=25000, sublinear_tf=True))
    ])

    X_train_feats = tfidf_union.fit_transform(X_train_text)
    X_val_feats = tfidf_union.transform(X_val_text)
    X_test_feats = tfidf_union.transform(X_test_text)

    rf_clf = RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    )
    rf_clf.fit(X_train_feats, y_train)

    val_preds = rf_clf.predict(X_val_feats)
    test_preds = rf_clf.predict(X_test_feats)

    val_macro_f1 = f1_score(y_val, val_preds, average="macro")
    test_macro_f1 = f1_score(y_test, test_preds, average="macro")
    test_acc = accuracy_score(y_test, test_preds)

    print(f"\nTask 1 Validation Macro-F1: {val_macro_f1:.4f}")
    print(f"Task 1 Test Macro-F1:       {test_macro_f1:.4f} (Accuracy: {test_acc:.4f})")
    print("\nTask 1 Detailed Test Classification Report:")
    print(classification_report(y_test, test_preds, target_names=["negative", "neutral", "positive"], digits=4))

    results = {
        "task1_lid_test_accuracy": float(lid_acc),
        "task1_lid_test_macro_f1": float(lid_f1),
        "task1_val_macro_f1": float(val_macro_f1),
        "task1_test_macro_f1": float(test_macro_f1),
        "task1_test_accuracy": float(test_acc)
    }
    with open(os.path.join(RESULTS_DIR, "task1_metrics.json"), "w") as f:
        json.dump(results, f, indent=2)

## Run

In [ ]:
main()

Metrics are in `results/task1_metrics.json`. Next: `task2/task2_muril.ipynb`.